# Limpieza de archivos Algoritmos

Cargar archivos de datos

In [2]:
from pathlib import Path
import pandas as pd
from utils_limpieza import *

archivos = sorted(Path("../data/algoritmosestructuras").glob("calificaciones*.xlsx"))
dfs = {a.name: pd.read_excel(a, header=6) for a in archivos}

Limpieza inicial de las calificaciones: nombres de columnas y filas nulas

In [3]:
for nombre, df in dfs.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    filas_nulas = df.isna().all(axis=1)
    sin_nombre = df[["apellido", "nombre"]].isna().any(axis=1) & ~filas_nulas
    sin_usuario = df["usuario"].isna() & ~filas_nulas
    print(nombre)
    print(f"  Filas totalmente nulas: {filas_nulas.sum()}")
    print(f"  Filas sin nombre/apellido: {sin_nombre.sum()}")
    print(f"  Filas sin usuario: {sin_usuario.sum()}")

    df = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])
    dfs[nombre] = filtrar_y_estandarizar_usuarios(df, nombre)

calificaciones_alumnos_20251024_115518.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115518.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115523.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115523.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115528.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115528.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115533.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115533.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115538.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115538.xlsx: staff/docentes excluidos = 0


Separación de la nota

In [4]:
NOTA_MAX = 100

for nombre, df in dfs.items():
    print(f"{nombre}: notas nulas = {df['nota'].isna().sum()}")
    df["nota"] = df["nota"].fillna(f"0/{NOTA_MAX}")
    df[["nota_obtenida", "nota_max"]] = df["nota"].str.split("/", expand=True).astype(float)
    dfs[nombre] = validar_notas(df, nombre, {NOTA_MAX})

calificaciones_alumnos_20251024_115518.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115518.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115523.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115523.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115528.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115528.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115533.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115533.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115538.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115538.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0


Eliminación de duplicados, unificación de calificaciones y exportación a Parquet

In [5]:
for nombre, df in dfs.items():
    duplicadas = df.duplicated().sum()
    usuarios_repetidos = df["usuario"].duplicated().sum()
    print(f"{nombre}: filas duplicadas = {duplicadas}, usuarios repetidos = {usuarios_repetidos}")

    dfs[nombre] = (df.drop_duplicates()
                   .sort_values("nota_obtenida", ascending=False)
                   .drop_duplicates(subset="usuario")
                   .sort_index())

df_calificaciones = pd.concat(dfs, names=["archivo"]).reset_index(level="archivo").reset_index(drop=True)
df_calificaciones = df_calificaciones.drop(columns=["nota"], errors="ignore")

df_calificaciones.to_parquet("../data/algoritmosestructuras/calificaciones_algoritmos.parquet", index=False)
df_calificaciones

calificaciones_alumnos_20251024_115518.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115523.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115528.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115533.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115538.xlsx: filas duplicadas = 0, usuarios repetidos = 0


,archivo,usuario,apellido,nombre,observaciones,nota_obtenida,nota_max
0,calificaciones_alumnos_20251024_115518.xlsx,ALU_1004,Flores,Gabriel,aprobada,100.0,100.0
1,calificaciones_alumnos_20251024_115518.xlsx,ALU_1005,Vega,Agustín,aprobada,100.0,100.0
2,calificaciones_alumnos_20251024_115518.xlsx,ALU_1006,Ramos,Patricio,aprobada,100.0,100.0
3,calificaciones_alumnos_20251024_115518.xlsx,ALU_1015,Ruiz,Gonzalo,aprobada,90.0,100.0
4,calificaciones_alumnos_20251024_115518.xlsx,ALU_1018,Rodríguez,Facundo,aprobada,70.0,100.0
...,...,...,...,...,...,...,...
414,calificaciones_alumnos_20251024_115538.xlsx,ALU_1274,Martínez,Santiago,aprobada,100.0,100.0
415,calificaciones_alumnos_20251024_115538.xlsx,ALU_1277,Silva,Gisela,aprobada,90.0,100.0
416,calificaciones_alumnos_20251024_115538.xlsx,ALU_1280,Sosa,Micaela,aprobada,90.0,100.0
417,calificaciones_alumnos_20251024_115538.xlsx,ALU_1293,Acosta,Diego,aprobada,100.0,100.0


Carga y limpieza del reporte de participación

In [6]:
archivo = next(Path("../data/algoritmosestructuras").glob("reporte_participacion*.xlsx"))
df_participacion = pd.read_excel(archivo, header=[5, 6])
# Encabezado de dos niveles (ej. "Encuestas" / "Cantidad"): se unen en un solo nombre
df_participacion.columns = pd.Index(
    [" ".join(n for n in col if not n.startswith("Unnamed")) for col in df_participacion.columns]
)

df_participacion = limpiar_reporte(df_participacion, "participacion")
df_participacion = procesar_participacion(df_participacion)

df_participacion.to_parquet("../data/algoritmosestructuras/participacion_algoritmos.parquet", index=False)
df_participacion

participacion: staff/docentes excluidos = 10
participacion:
  Fechas inconsistentes (última < primera, o primera < alta): 0
  Filas con valores numéricos negativos: 0
  Avance fuera de 0-100: 0
  Aulas: ['TUCD-MD-Algoritmos y Estructuras de Datos 2025']
  Perfiles: {'Alumnos': 319}
  Filas duplicadas: 0, usuarios repetidos: 0
  Nulos por columna: {'primera_interaccion': 43, 'ultima_interaccion': 43}


,usuario,apellido,nombre,aula,perfil,fecha_alta,avance,interacciones,primera_interaccion,ultima_interaccion,...,foros_temas_de_debate_con_intervencion,foros_cantidad_de_intervenciones_del_usuario,anuncios_cantidad,anuncios_cargados,noticias_cantidad,noticias_cargados,tiempo_estimado_de_uso_min,fechas_inconsistentes,valores_negativos,avance_fuera_de_rango
0,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,47,406,2025-08-05 18:05:40,2025-10-21 20:15:05,...,0,0,0,0,6,0,203.0,False,False,False
1,ALU_1002,Silva,Diego,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,87,445,2025-08-05 14:21:32,2025-10-18 13:42:51,...,0,0,0,0,6,0,284.0,False,False,False
2,ALU_1003,Castro,Federico,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,0,1,2025-08-06 20:03:24,2025-08-06 20:03:24,...,0,0,0,0,6,0,0.0,False,False,False
3,ALU_1004,Flores,Gabriel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,100,865,2025-08-05 11:33:48,2025-10-22 09:50:49,...,0,0,0,0,6,0,572.0,False,False,False
4,ALU_1005,Vega,Agustín,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,98,420,2025-08-05 21:39:15,2025-10-24 10:09:36,...,0,0,0,0,6,0,259.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,ALU_1315,Flores,Felipe,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,98,356,2025-08-05 22:52:19,2025-10-21 20:14:15,...,0,0,0,0,6,0,108.0,False,False,False
315,ALU_1316,Sosa,Emilia,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,32,182,2025-08-05 19:54:37,2025-10-19 22:29:17,...,0,0,0,0,6,0,34.0,False,False,False
316,ALU_1317,Morales,Nahuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,0,10,2025-08-25 18:51:19,2025-09-18 10:42:09,...,0,0,0,0,6,0,1.0,False,False,False
317,ALU_1318,Villalba,Candelaria,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,2025-08-05,35,104,2025-08-05 18:16:29,2025-09-15 17:58:34,...,0,0,0,0,6,0,81.0,False,False,False


Carga y limpieza del reporte de trazabilidad

In [7]:
archivo = next(Path("../data/algoritmosestructuras").glob("reporte_trazabilidad*.xlsx"))
df_trazabilidad = procesar_trazabilidad(limpiar_reporte(pd.read_excel(archivo, header=5), "trazabilidad"))

df_trazabilidad.to_parquet("../data/algoritmosestructuras/trazabilidad_algoritmos.parquet", index=False)
df_trazabilidad

trazabilidad: staff/docentes excluidos = 1408
trazabilidad:
  Eventos sin fecha válida (descartados): 0
  Rango de fechas: 2025-08-05 10:18:05 -> 2025-10-24 11:53:27
  Aulas: ['TUCD-MD-Algoritmos y Estructuras de Datos 2025']
  Perfiles: {'Alumnos': 44543}
  Filas duplicadas (descartadas): 171
  Nulos por columna: ninguno


,fecha,ip_usuario,usuario,apellido,nombre,aula,perfil,seccion,accion,descripcion
0,2025-08-05 18:05:40,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'
1,2025-08-05 18:25:40,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
2,2025-08-05 18:25:55,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Clase 1  Notebook: Hola Mund...
3,2025-08-05 18:27:45,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
4,2025-08-05 18:27:50,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Hoja de Ruta  Unidad 1' de l...
...,...,...,...,...,...,...,...,...,...,...
44367,2025-10-21 20:05:09,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Clase 10: Diccionarios' de la...
44368,2025-10-21 20:05:18,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
44369,2025-10-21 20:06:23,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Clase 10: Actividad no entreg...
44370,2025-10-21 20:07:07,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Algoritmos y Estructuras de Datos 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'


Cruce con Matriculados

In [8]:
df_matriculados = pd.read_parquet("../data/matriculados/matriculados.parquet")
df_cruce = cruzar_con_matriculados(df_matriculados, "algoritmos", {
    "trazabilidad": df_trazabilidad,
    "participacion": df_participacion,
    "calificaciones": df_calificaciones,
})
df_cruce.to_parquet("../data/algoritmosestructuras/cruce_matriculados_algoritmos.parquet", index=False)
df_cruce[df_cruce["activo_no_inscripto"]]

Matriculados (algoritmos): 167
  trazabilidad: 276 usuarios, 111 no matriculados
  participacion: 319 usuarios, 152 no matriculados
  calificaciones: 146 usuarios, 39 no matriculados
Activos no inscriptos: 111
Activos con estado Baja: 2
Inscriptos sin actividad: 2


,usuario,matriculado,en_trazabilidad,en_participacion,en_calificaciones,estado_insc,instancia,activo_no_inscripto,activo_con_baja,inscripto_sin_actividad
2,ALU_1003,False,True,True,False,<NA>,<NA>,True,False,False
4,ALU_1005,False,True,True,True,<NA>,<NA>,True,False,False
6,ALU_1007,False,True,True,False,<NA>,<NA>,True,False,False
8,ALU_1009,False,True,True,False,<NA>,<NA>,True,False,False
10,ALU_1011,False,True,True,False,<NA>,<NA>,True,False,False
...,...,...,...,...,...,...,...,...,...,...
312,ALU_1313,False,True,True,False,<NA>,<NA>,True,False,False
314,ALU_1315,False,True,True,True,<NA>,<NA>,True,False,False
316,ALU_1317,False,True,True,False,<NA>,<NA>,True,False,False
317,ALU_1318,False,True,True,False,<NA>,<NA>,True,False,False


Verificar nulos y observaciones

In [9]:
sin_fecha = df_participacion["primera_interaccion"].isna()
print("Sin fecha pero con interacciones:", int((sin_fecha & (df_participacion["interacciones"] > 0)).sum()))
print("Con fecha pero 0 interacciones:", int((~sin_fecha & (df_participacion["interacciones"] == 0)).sum()))

con_actividad = set(df_participacion.loc[df_participacion["interacciones"] > 0, "usuario"])
print("Con interacciones pero sin trazabilidad:", len(con_actividad - set(df_trazabilidad["usuario"])))
print("En trazabilidad pero con 0 interacciones:", len(set(df_trazabilidad["usuario"]) - con_actividad))

print(df_calificaciones["observaciones"].value_counts(dropna=False))

Sin fecha pero con interacciones: 0
Con fecha pero 0 interacciones: 0
Con interacciones pero sin trazabilidad: 0
En trazabilidad pero con 0 interacciones: 0
observaciones
aprobada       418
no aprobada      1
Name: count, dtype: int64
